# Chương 7 – Knowledge Distillation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter07/chapter07_colab.ipynb)

Teacher = model SFT chương 6 (tải từ Hugging Face). Student = Qwen3-4B + LoRA, train trên output
của teacher. Phần bonus so sánh model SFT với **Claude** (thay cho OpenRouter trong sách).

**GPU:** L4 (24 GB) hoặc A100.

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

In [ ]:
import os, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab
# Sách chỉ dùng PyTorch. Không cho transformers nạp TensorFlow có sẵn trên Colab
# (tránh cảnh báo giả "Could not find cuda drivers" của TensorFlow).
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

### Tải model đã train sẵn từ Hugging Face

Tác giả sách đã publish model của từng chương tại [`bahree/ModelAdaptationBook`](https://huggingface.co/bahree/ModelAdaptationBook). Ô dưới tải đúng các thư mục cần cho chương này về `/content/hub/` để các script dùng như đường dẫn local (không cần tự train các chương trước).

In [ ]:
from huggingface_hub import snapshot_download

HUB_DIR = "/content/hub"
SUBFOLDERS = ["ch6-sft"]
snapshot_download("bahree/ModelAdaptationBook", local_dir=HUB_DIR,
                  allow_patterns=[f"{s}/*" for s in SUBFOLDERS])
for s in SUBFOLDERS:
    print(s, "->", f"{HUB_DIR}/{s}", sorted(os.listdir(f"{HUB_DIR}/{s}"))[:6])

In [ ]:
TEACHER_DIR = "/content/hub/ch6-sft"

## 1. Unit test

In [ ]:
run("pytest chapter07/tests/ -q")

## 2. Stage 1 – Sinh dữ liệu từ teacher (Listing 7.1, tuỳ chọn)

Output đầy đủ 200 prompt đã commit sẵn trong `chapter07/data/teacher_outputs.jsonl`. Ô dưới chỉ chạy thử 20 prompt ra file riêng để bạn thấy cách hoạt động.

In [ ]:
run(f"python -m chapter07.scripts.generate_teacher_data --teacher_dir {TEACHER_DIR} "
    "--prompts data/it_support_fmt/train.jsonl "
    "--out chapter07/data/teacher_outputs_colab_sample.jsonl --num_prompts 20")

## 3. Stage 2 – Lọc chất lượng + chia train/valid (Listing 7.2)

In [ ]:
run("python -m chapter07.scripts.prepare_distillation_data "
    "--input chapter07/data/teacher_outputs.jsonl "
    "--out chapter07/data/distill_ready_colab --train 160 --valid 40")

## 4. Stage 3 – Train student LoRA (Listing 7.3)

~3–5 phút.

In [ ]:
run("python -m chapter07.train_student "
    "--train chapter07/data/distill_ready_colab/train.jsonl "
    "--valid chapter07/data/distill_ready_colab/valid.jsonl "
    "--out chapter07/runs/student_run1")

## 5. Stage 4 – Đánh giá 3 chiều base / teacher / student (Listing 7.4)

~20–40 phút.

In [ ]:
run(f"python -m chapter07.eval_distillation --data_dir chapter07/data/distill_ready_colab "
    f"--teacher_dir {TEACHER_DIR} --student_dir chapter07/runs/student_run1 "
    "--source_dir data/it_support_fmt --reference human "
    "--eval_file data/it_support/test.jsonl "
    "--output chapter07/eval/test_split/distillation_eval_colab.json")

In [ ]:
import json, pathlib
p = pathlib.Path("chapter07/eval/test_split/distillation_eval_colab.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:3000] if p.exists() else f"Chưa có {p}")

## 6. Stage 5 – Safety robustness check (Listing 7.5)

In [ ]:
run("python -m chapter07.scripts.robustness_check --student_dir chapter07/runs/student_run1 --output_dir chapter07/eval/safety_colab")

## 7. Inference với student

In [ ]:
run('python -m chapter07.generate --adapter_dir chapter07/runs/student_run1 --prompt "How do I troubleshoot a VPN connection failure?"')

## 8. (Bonus) So sánh model SFT với Claude

Script `capture_frontier_comparison` vốn gọi OpenRouter. Ô dưới tạo `/content/run_with_claude.py` để chuyển các lời gọi đó sang Claude API bằng key của bạn.

In [ ]:
%%writefile /content/run_with_claude.py
"""Chạy một module của sách nhưng thay OpenRouter bằng Claude API (ANTHROPIC_API_KEY).

Dùng: python /content/run_with_claude.py <module> [args...]
Các script trong sách gọi common.openrouter.chat(); file này thay hàm đó bằng một hàm
gọi Anthropic SDK rồi chạy module như `python -m <module>`.
"""
import os
import runpy
import sys

sys.path.insert(0, os.getcwd())

import anthropic
import common.openrouter as openrouter

# Model Claude dùng cho mọi lời gọi không chỉ định rõ model Anthropic
# (ví dụ judge "openai/gpt-5.5" trong chương 8). Đổi qua biến môi trường CLAUDE_MODEL.
CLAUDE_MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")
_client = anthropic.Anthropic()


def _resolve(model):
    # "anthropic/claude-sonnet-5.5" -> "claude-sonnet-5-5"; model của hãng khác -> CLAUDE_MODEL
    if model and model.startswith("anthropic/"):
        return model.split("/", 1)[1].replace(".", "-")
    return CLAUDE_MODEL


def chat(messages, model, *, max_tokens=1024, temperature=0.0, timeout=60, extra_body=None):
    # temperature bị bỏ qua: các model Claude hiện tại không nhận tham số sampling.
    system = "\n\n".join(m["content"] for m in messages if m["role"] == "system")
    convo = [m for m in messages if m["role"] != "system"]
    kwargs = {"system": system} if system else {}
    resp = _client.messages.create(
        model=_resolve(model),
        max_tokens=max(max_tokens, 4096),  # chừa chỗ cho adaptive thinking
        output_config={"effort": "low"},
        messages=convo,
        **kwargs,
    )
    text = "".join(b.text for b in resp.content if b.type == "text")
    if not text:
        raise openrouter.OpenRouterError(f"Claude trả về rỗng (stop_reason={resp.stop_reason})")
    u = resp.usage
    return {
        "content": text,
        "reasoning": None,
        "usage": {"prompt_tokens": u.input_tokens, "completion_tokens": u.output_tokens,
                  "total_tokens": u.input_tokens + u.output_tokens, "cost": None},
        "model": resp.model,
        "raw": resp.to_dict(),
    }


openrouter.chat = chat
os.environ.setdefault("OPENROUTER_API_KEY", "not-used-claude-shim")

module = sys.argv[1]
sys.argv = [module] + sys.argv[2:]
runpy.run_module(module, run_name="__main__", alter_sys=True)

In [ ]:
run(f"python /content/run_with_claude.py chapter07.scripts.capture_frontier_comparison "
    f"--sft_dir {TEACHER_DIR} "
    "--frontier_models anthropic/claude-sonnet-5.5 anthropic/claude-opus-5.5 anthropic/claude-haiku-5.5 "
    "--output chapter07/runs/frontier_comparison_claude.json")

## Lưu kết quả về Google Drive (tuỳ chọn)

Runtime Colab sẽ bị xoá khi ngắt kết nối. Chạy ô dưới để copy thư mục kết quả sang Drive.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/ModelAdaptationBook"
for p in ["chapter07/runs", "chapter07/eval/test_split", "chapter07/eval/safety_colab"]:
    src = os.path.join(CODE_DIR, p)
    dst = os.path.join(DEST, p)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Đã lưu", p, "->", dst)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Đã lưu", p, "->", dst)